In [4]:
import pandas as pd 

df = pd.read_csv("deployment_gate_training_v4.csv")
df = df.fillna(0)

print(df.shape)
df.head()

(800, 18)


,pipeline_id,total_tasks,is_production,pipeline_success,failed_tasks,stage_count,task_failure_rate,project_age_days,days_since_last_push,stars_to_forks_ratio,build_tool_count,uses_legacy_build,uses_multiple_ides,uses_ci_and_submodules,avg_file_churn,new_file_ratio,dependency_error_rate,compiler_error_rate
0,pipe-aajpt,1,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,pipe-aavnq,1,0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,pipe-acftp,1,0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,pipe-acoca,1,0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,pipe-adxsr,1,0,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
df["high_risk"] = (df["pipeline_success"]==0).astype(int)

df["high_risk"].value_counts(normalize=True)

high_risk
0    0.7575
1    0.2425
Name: proportion, dtype: float64

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

X = df.drop(columns=["pipeline_id","pipeline_success","high_risk"])
Y= df["high_risk"]

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1

)
scores = cross_val_score(rf,X,Y,cv=5,scoring="roc_auc")
print("AUC scores",scores.mean())




AUC scores 0.5273561379922663


In [7]:
rf.fit(X,Y)
df["risk_score"] = rf.predict_proba(X)[:,1]
df[["risk_score"]].describe()

,risk_score
count,800.000000
mean,0.499908
std,0.026115
min,0.453782
25%,0.514674
50%,0.514674
75%,0.514674
max,0.514674


In [8]:
import pandas as pd 
import numpy as np

p80 = np.percentile(df["risk_score"],80)
p60 = np.percentile(df["risk_score"],60)

def deployment_gate(row):
    if row["is_production"]==1 and row["risk_score"]>=p80:
        return "BLOCK"
    elif row["risk_score"]>=p60:
        return "WARN"
    return "ALLOW"

df["gate_decision"] = df.apply(deployment_gate,axis=1)
df["gate_decision"].value_counts()



gate_decision
WARN     606
ALLOW    194
Name: count, dtype: int64

In [9]:
from sklearn.metrics import recall_score

failed = df["pipeline_success"] == 0
flagged = df["gate_decision"].isin(["BLOCK", "WARN"])

print(
    "Failure recall (BLOCK + WARN):",
    recall_score(failed, flagged)
)


Failure recall (BLOCK + WARN): 0.7989690721649485


In [10]:
type(rf)

import joblib
from pathlib import Path

Path("model").mkdir(exist_ok=True)
joblib.dump(rf, "model/deployment_gate_model.joblib")
print("model saved")

model saved


In [11]:
import joblib 
import pandas as pd

loaded_model = joblib.load("model/deployment_gate_model.joblib")

sample =X.iloc[[0]]
print(loaded_model.predict_proba(sample))



[[0.48532595 0.51467405]]


In [12]:
joblib.dump(rf, "model/rf_model.joblib")


['model/rf_model.joblib']

In [14]:
import joblib
from pathlib import Path

Path("model").mkdir(exist_ok=True)

bundle = {
    "model": rf,
    "features" : X.columns.tolist()
}


joblib.dump(bundle, "model/deployment_gate_model.joblib")

print("✅ Model bundle saved successfully")


✅ Model bundle saved successfully


In [16]:
X.columns.tolist()

['total_tasks',
 'is_production',
 'failed_tasks',
 'stage_count',
 'task_failure_rate',
 'project_age_days',
 'days_since_last_push',
 'stars_to_forks_ratio',
 'build_tool_count',
 'uses_legacy_build',
 'uses_multiple_ides',
 'uses_ci_and_submodules',
 'avg_file_churn',
 'new_file_ratio',
 'dependency_error_rate',
 'compiler_error_rate']